# Generate Full and Evidence-Grounded Candidate Banks for DPO

Sample the evidence-grounded Qwen 2.5 0.5B SFT adapter on all 1,600 factoid questions once. Separate evidence-only banks are then filtered from those exact candidates, without a second generation pass.


In [ ]:
import json
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / 'Task-Structured Counterfactual Preference Mining']
project_root = next((path for path in candidates if (path / 'src').is_dir()), None)
assert project_root is not None, 'Run this notebook from the project or workspace directory.'

MODEL_REF = project_root / 'Artifacts/Factoid_SFT/models/evidence_grounded_single_answer_qwen25_05b/adapter'
FULL_SOURCE_DIR = project_root / 'data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b'
TRAIN_INPUT = FULL_SOURCE_DIR / 'train_prepared.json'
DEV_INPUT = FULL_SOURCE_DIR / 'eval_prepared.json'
SUPPORTED_DIR = project_root / 'data/BioASQ_factoid_sft_prepared/evidence_grounded_single_answer_qwen25_05b'
SUPPORTED_TRAIN_INPUT = SUPPORTED_DIR / 'train_prepared.json'
SUPPORTED_DEV_IDS = SUPPORTED_DIR / 'dev_supported_question_ids.json'
FULL_TRAIN_OUTPUT_ROOT = project_root / 'Artifacts/cse_dpo/candidate_banks/factoid_full_resources_single_answer_evidence_grounded_qwen25_05b_train_s3_n20'
FULL_DEV_OUTPUT_ROOT = project_root / 'Artifacts/cse_dpo/candidate_banks/factoid_full_resources_single_answer_evidence_grounded_qwen25_05b_dev_s3_n20'
SUPPORTED_TRAIN_OUTPUT_ROOT = project_root / 'Artifacts/cse_dpo/candidate_banks/factoid_evidence_only_single_answer_qwen25_05b_train_s3_n20'
SUPPORTED_DEV_OUTPUT_ROOT = project_root / 'Artifacts/cse_dpo/candidate_banks/factoid_evidence_only_single_answer_qwen25_05b_dev_s3_n20'
MODEL_SLUG = 'evidence-grounded-single-answer-qwen25-05b'
SAMPLES_PER_QUESTION = 20
SKIP_COMPLETE_SPLITS = True
SPLITS = {
    'train': {'input': TRAIN_INPUT, 'output_root': FULL_TRAIN_OUTPUT_ROOT},
    'dev': {'input': DEV_INPUT, 'output_root': FULL_DEV_OUTPUT_ROOT},
}

for path in (MODEL_REF / 'adapter_config.json', TRAIN_INPUT, DEV_INPUT, SUPPORTED_TRAIN_INPUT, SUPPORTED_DEV_IDS):
    assert path.exists(), f'Missing required input: {path}'

print(f'Model: {MODEL_REF}')
print('Full generation: 1,440 train questions and 160 dev questions.')


In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n')


supported_train_ids = {row['id'] for row in json.loads(SUPPORTED_TRAIN_INPUT.read_text(encoding='utf-8'))}
supported_dev_ids = set(json.loads(SUPPORTED_DEV_IDS.read_text(encoding='utf-8'))['question_ids'])
assert len(supported_train_ids) == 1130, len(supported_train_ids)
assert len(supported_dev_ids) == 129, len(supported_dev_ids)
print(f'Evidence-only views: {len(supported_train_ids)} train questions, {len(supported_dev_ids)} dev questions.')


## Sample Full Candidate Banks

This is the only generation step. It creates 28,800 train candidates and 3,200 dev candidates.


In [ ]:
import subprocess


def completed_candidate_bank(split_config: dict) -> bool:
    candidate_bank = split_config['output_root'] / MODEL_SLUG / 'candidate_bank.jsonl'
    manifest_path = split_config['output_root'] / MODEL_SLUG / 'manifest.json'
    if not candidate_bank.exists() or not manifest_path.exists():
        return False
    expected_questions = len(json.loads(split_config['input'].read_text(encoding='utf-8')))
    expected_candidates = expected_questions * SAMPLES_PER_QUESTION
    try:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        actual_candidates = sum(1 for line in candidate_bank.open(encoding='utf-8') if line.strip())
    except (OSError, json.JSONDecodeError):
        return False
    return (
        manifest.get('dataset', {}).get('question_count') == expected_questions
        and manifest.get('candidate_count') == expected_candidates
        and actual_candidates == expected_candidates
    )


def generate_split(split_name: str, split_config: dict):
    command = [
        'python', '-m', 'cse_dpo.generate_candidate_bank',
        '--eval-input', str(split_config['input']),
        '--model-ref', str(MODEL_REF),
        '--output-dir', str(split_config['output_root']),
        '--dataset-name', 'bioasq-factoid',
        '--question-types', 'factoid',
        '--prompt-file', 'prompts/factoid_single_answer_aligned.json',
        '--prompt', 'factoid-single-answer-aligned-v1',
        '--chat-template', 'qwen-2.5',
        '--prompt-format', 'chat',
        '--samples-per-question-total', str(SAMPLES_PER_QUESTION),
        '--max-resources', '0',
        '--max-resource-chars', '0',
        '--resource-selection', 'first',
        '--resource-granularity', 'document',
        '--max-seq-length', '9000',
        '--max-new-tokens', '64',
        '--temperature', '0.7',
        '--top-p', '0.9',
        '--batch-size', '1',
        '--seed', '3407',
        '--factoid-parser-mode', 'agnostic',
        '--local-files-only',
    ]
    print(f'Generating {split_name} bank:')
    print(' '.join(command))
    subprocess.run(command, cwd=project_root, check=True)


for split_name, split_config in SPLITS.items():
    if SKIP_COMPLETE_SPLITS and completed_candidate_bank(split_config):
        print(f'Skipping complete {split_name} bank.')
        continue
    generate_split(split_name, split_config)


## Filter Evidence-Only Banks

This copies candidate rows from the full banks, preserving identical candidates for overlapping questions.


In [ ]:
def filter_bank(split_name: str, supported_ids: set[str], filtered_root: Path) -> dict:
    full_root = SPLITS[split_name]['output_root']
    full_bank = full_root / MODEL_SLUG / 'candidate_bank.jsonl'
    full_manifest_path = full_root / MODEL_SLUG / 'manifest.json'
    assert full_bank.exists() and full_manifest_path.exists(), f'Generate the full {split_name} bank first.'
    rows = [row for row in load_jsonl(full_bank) if row.get('question_id') in supported_ids]
    expected_candidates = len(supported_ids) * SAMPLES_PER_QUESTION
    assert len(rows) == expected_candidates, (split_name, len(rows), expected_candidates)
    target_dir = filtered_root / MODEL_SLUG
    target_bank = target_dir / 'candidate_bank.jsonl'
    target_manifest = target_dir / 'manifest.json'
    write_jsonl(target_bank, rows)
    manifest = json.loads(full_manifest_path.read_text(encoding='utf-8'))
    manifest['dataset']['question_count'] = len(supported_ids)
    manifest['candidate_count'] = len(rows)
    manifest['filtered_from_candidate_bank'] = str(full_bank)
    manifest['evidence_grounded_only'] = True
    target_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    return {'questions': len(supported_ids), 'candidates': len(rows), 'path': target_bank}


train_filtered = filter_bank('train', supported_train_ids, SUPPORTED_TRAIN_OUTPUT_ROOT)
dev_filtered = filter_bank('dev', supported_dev_ids, SUPPORTED_DEV_OUTPUT_ROOT)
print({'evidence_only_train': train_filtered, 'evidence_only_dev': dev_filtered})
